<a href="https://colab.research.google.com/github/kybertechnologies/kyber-atlas-prototype/blob/main/atlas_build.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas
pandas_version = pandas.__version__
!pip install -q pypsa networkx plotly scikit-learn "pandas=={pandas_version}" numpy streamlit matplotlib

import pypsa, networkx, plotly, sklearn, pandas, numpy, streamlit, matplotlib; print("All libraries loaded successfully")

ModuleNotFoundError: No module named 'pandas'

In [ ]:
import subprocess
import sys

try:
    import pypsa
except ImportError:
    from importlib.metadata import version

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "pypsa", f"pandas=={version('pandas')}"]
    )
    import pypsa

import logging

import pandas as pd

logging.getLogger("pypsa").setLevel(logging.WARNING)
pd.set_option("display.width", 120)


def build_ieee_9bus() -> pypsa.Network:
    """IEEE 9-bus (WSCC) system using MATPOWER case9 data, 100 MVA base, 345 kV."""
    base_mva, v_nom = 100.0, 345.0
    z_base = v_nom**2 / base_mva

    n = pypsa.Network(name="IEEE 9-bus")
    for i in range(1, 10):
        n.add("Bus", f"Bus {i}", v_nom=v_nom)

    # (from, to, r pu, x pu, b pu)
    branches = [
        (1, 4, 0.0, 0.0576, 0.0),
        (4, 5, 0.017, 0.092, 0.158),
        (5, 6, 0.039, 0.17, 0.358),
        (3, 6, 0.0, 0.0586, 0.0),
        (6, 7, 0.0119, 0.1008, 0.209),
        (7, 8, 0.0085, 0.072, 0.149),
        (8, 2, 0.0, 0.0625, 0.0),
        (8, 9, 0.032, 0.161, 0.306),
        (9, 4, 0.01, 0.085, 0.176),
    ]
    for f, t, r, x, b in branches:
        n.add("Line", f"Line {f}-{t}", bus0=f"Bus {f}", bus1=f"Bus {t}",
              r=r * z_base, x=x * z_base, b=b / z_base, s_nom=250.0)

    n.add("Generator", "Gen 1", bus="Bus 1", control="Slack", p_set=71.6)
    n.add("Generator", "Gen 2", bus="Bus 2", control="PV", p_set=163.0)
    n.add("Generator", "Gen 3", bus="Bus 3", control="PV", p_set=85.0)

    n.add("Load", "Load 5", bus="Bus 5", p_set=90.0, q_set=30.0)
    n.add("Load", "Load 7", bus="Bus 7", p_set=100.0, q_set=35.0)
    n.add("Load", "Load 9", bus="Bus 9", p_set=125.0, q_set=50.0)
    return n


def title(text: str) -> None:
    print(f"\n{text}\n{'=' * len(text)}")


if hasattr(pypsa.examples, "ac_9bus"):
    network = pypsa.examples.ac_9bus()
else:
    print(f"pypsa {pypsa.__version__} has no examples.ac_9bus(); building the IEEE 9-bus system from MATPOWER case9 data.")
    network = build_ieee_9bus()

title("Buses")
buses = network.buses[["v_nom"]].rename(columns={"v_nom": "Base voltage (kV)"})
buses.index.name = "Bus"
print(buses.to_string(float_format="{:.1f}".format))

title("Transmission lines")
lines = network.lines[["bus0", "bus1", "r", "x"]].rename(
    columns={"bus0": "From bus", "bus1": "To bus", "r": "Resistance (ohm)", "x": "Reactance (ohm)"}
)
lines.index.name = "Line"
print(lines.to_string(float_format="{:.3f}".format))

network.lpf()

title("Linear power flow results")
snapshot = network.snapshots[0]
results = pd.DataFrame({
    "Voltage magnitude (pu)": network.buses_t.v_mag_pu.loc[snapshot],
    "Voltage angle (deg)": network.buses_t.v_ang.loc[snapshot] * 180 / 3.141592653589793,
})
results.index.name = "Bus"
print(results.to_string(float_format="{:.4f}".format))
print(
    "\nNote: linear (DC) power flow assumes every bus is at 1.0 pu and only solves for voltage angles.\n"
    "Run network.pf() for a full AC power flow if you need real voltage magnitudes."
)

title("Network summary")
print(f"Buses:      {len(network.buses)}")
print(f"Lines:      {len(network.lines)}")
print(f"Generators: {len(network.generators)}")

pypsa 1.2.4 has no examples.ac_9bus(); building the IEEE 9-bus system from MATPOWER case9 data.

Buses
=====
       Base voltage (kV)
Bus                     
Bus 1              345.0
Bus 2              345.0
Bus 3              345.0
Bus 4              345.0
Bus 5              345.0
Bus 6              345.0
Bus 7              345.0
Bus 8              345.0
Bus 9              345.0

Transmission lines
         From bus To bus  Resistance (ohm)  Reactance (ohm)
Line                                                       
Line 1-4    Bus 1  Bus 4             0.000           68.558
Line 4-5    Bus 4  Bus 5            20.234          109.503
Line 5-6    Bus 5  Bus 6            46.420          202.343
Line 3-6    Bus 3  Bus 6             0.000           69.749
Line 6-7    Bus 6  Bus 7            14.164          119.977
Line 7-8    Bus 7  Bus 8            10.117           85.698
Line 8-2    Bus 8  Bus 2             0.000           74.391
Line 8-9    Bus 8  Bus 9            38.088          191.

In [ ]:
import subprocess
import sys

try:
    import pypsa
except ImportError:
    from importlib.metadata import version

    # Keep Colab's pinned pandas so pip picks a compatible pypsa instead of upgrading pandas.
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "pypsa", f"pandas=={version('pandas')}"]
    )
    import pypsa

import logging

import numpy as np
import pandas as pd

logging.getLogger("pypsa").setLevel(logging.WARNING)
pd.set_option("display.width", 120)

BASE_MVA = 100.0


def add_branches(n: pypsa.Network, branches: list, v_nom: float) -> None:
    """Add MATPOWER-style branches (from, to, r pu, x pu, b pu, tap ratio).

    A tap ratio of 0 means a line; anything else is a transformer.
    """
    z_base = v_nom**2 / BASE_MVA
    for f, t, r, x, b, ratio in branches:
        if ratio == 0:
            n.add("Line", f"Line {f}-{t}", bus0=f"Bus {f}", bus1=f"Bus {t}",
                  r=r * z_base, x=x * z_base, b=b / z_base, s_nom=250.0)
        else:
            n.add("Transformer", f"Trafo {f}-{t}", bus0=f"Bus {f}", bus1=f"Bus {t}",
                  r=r, x=x, b=b, s_nom=BASE_MVA, tap_ratio=ratio)


def build_ieee_9bus() -> pypsa.Network:
    """IEEE 9-bus (WSCC) system from MATPOWER case9, 100 MVA base, 345 kV."""
    v_nom = 345.0
    n = pypsa.Network(name="IEEE 9-bus")
    for i in range(1, 10):
        n.add("Bus", f"Bus {i}", v_nom=v_nom)

    add_branches(n, [
        (1, 4, 0.0, 0.0576, 0.0, 0),
        (4, 5, 0.017, 0.092, 0.158, 0),
        (5, 6, 0.039, 0.17, 0.358, 0),
        (3, 6, 0.0, 0.0586, 0.0, 0),
        (6, 7, 0.0119, 0.1008, 0.209, 0),
        (7, 8, 0.0085, 0.072, 0.149, 0),
        (8, 2, 0.0, 0.0625, 0.0, 0),
        (8, 9, 0.032, 0.161, 0.306, 0),
        (9, 4, 0.01, 0.085, 0.176, 0),
    ], v_nom)

    # (bus, control, P MW, voltage setpoint pu)
    for bus, control, p, v in [(1, "Slack", 71.6, 1.0), (2, "PV", 163.0, 1.0), (3, "PV", 85.0, 1.0)]:
        n.add("Generator", f"Gen {bus}", bus=f"Bus {bus}", control=control, p_set=p)
        n.buses.loc[f"Bus {bus}", "v_mag_pu_set"] = v

    for bus, p, q in [(5, 90.0, 30.0), (7, 100.0, 35.0), (9, 125.0, 50.0)]:
        n.add("Load", f"Load {bus}", bus=f"Bus {bus}", p_set=p, q_set=q)
    return n


def build_ieee_14bus() -> pypsa.Network:
    """IEEE 14-bus system from MATPOWER case14, 100 MVA base, all buses at 135 kV."""
    v_nom = 135.0
    n = pypsa.Network(name="IEEE 14-bus")
    for i in range(1, 15):
        n.add("Bus", f"Bus {i}", v_nom=v_nom)

    add_branches(n, [
        (1, 2, 0.01938, 0.05917, 0.0528, 0),
        (1, 5, 0.05403, 0.22304, 0.0492, 0),
        (2, 3, 0.04699, 0.19797, 0.0438, 0),
        (2, 4, 0.05811, 0.17632, 0.034, 0),
        (2, 5, 0.05695, 0.17388, 0.0346, 0),
        (3, 4, 0.06701, 0.17103, 0.0128, 0),
        (4, 5, 0.01335, 0.04211, 0.0, 0),
        (4, 7, 0.0, 0.20912, 0.0, 0.978),
        (4, 9, 0.0, 0.55618, 0.0, 0.969),
        (5, 6, 0.0, 0.25202, 0.0, 0.932),
        (6, 11, 0.09498, 0.1989, 0.0, 0),
        (6, 12, 0.12291, 0.25581, 0.0, 0),
        (6, 13, 0.06615, 0.13027, 0.0, 0),
        (7, 8, 0.0, 0.17615, 0.0, 0),
        (7, 9, 0.0, 0.11001, 0.0, 0),
        (9, 10, 0.03181, 0.0845, 0.0, 0),
        (9, 14, 0.12711, 0.27038, 0.0, 0),
        (10, 11, 0.08205, 0.19207, 0.0, 0),
        (12, 13, 0.22092, 0.19988, 0.0, 0),
        (13, 14, 0.17093, 0.34802, 0.0, 0),
    ], v_nom)

    # Generators at buses 3, 6 and 8 are synchronous condensers (0 MW, voltage support only).
    for bus, control, p, v in [
        (1, "Slack", 232.4, 1.06),
        (2, "PV", 40.0, 1.045),
        (3, "PV", 0.0, 1.01),
        (6, "PV", 0.0, 1.07),
        (8, "PV", 0.0, 1.09),
    ]:
        n.add("Generator", f"Gen {bus}", bus=f"Bus {bus}", control=control, p_set=p)
        n.buses.loc[f"Bus {bus}", "v_mag_pu_set"] = v

    for bus, p, q in [
        (2, 21.7, 12.7), (3, 94.2, 19.0), (4, 47.8, -3.9), (5, 7.6, 1.6),
        (6, 11.2, 7.5), (9, 29.5, 16.6), (10, 9.0, 5.8), (11, 3.5, 1.8),
        (12, 6.1, 1.6), (13, 13.5, 5.8), (14, 14.9, 5.0),
    ]:
        n.add("Load", f"Load {bus}", bus=f"Bus {bus}", p_set=p, q_set=q)

    n.add("ShuntImpedance", "Shunt 9", bus="Bus 9", b=19.0 / v_nom**2)
    return n


def load_network(example_name: str, fallback) -> pypsa.Network:
    if hasattr(pypsa.examples, example_name):
        return getattr(pypsa.examples, example_name)()
    print(f"pypsa {pypsa.__version__} has no examples.{example_name}(); building it from MATPOWER data.")
    return fallback()


def title(text: str, char: str = "=") -> None:
    print(f"\n{text}\n{char * len(text)}")


def analyze(network: pypsa.Network) -> dict:
    title(f"{network.name}", "#")

    title("Buses")
    buses = network.buses[["v_nom"]].rename(columns={"v_nom": "Base voltage (kV)"})
    buses.index.name = "Bus"
    print(buses.to_string(float_format="{:.1f}".format))

    title("Transmission lines")
    lines = network.lines[["bus0", "bus1", "r", "x"]].rename(
        columns={"bus0": "From bus", "bus1": "To bus", "r": "Resistance (ohm)", "x": "Reactance (ohm)"}
    )
    lines.index.name = "Line"
    print(lines.to_string(float_format="{:.3f}".format))

    if len(network.transformers):
        title("Transformers")
        trafos = network.transformers[["bus0", "bus1", "r", "x", "tap_ratio"]].rename(
            columns={"bus0": "From bus", "bus1": "To bus", "r": "Resistance (pu)",
                     "x": "Reactance (pu)", "tap_ratio": "Tap ratio"}
        )
        trafos.index.name = "Transformer"
        print(trafos.to_string(float_format="{:.4f}".format))

    network.lpf()

    title("Linear power flow results")
    snapshot = network.snapshots[0]
    angles_deg = np.degrees(network.buses_t.v_ang.loc[snapshot])
    results = pd.DataFrame({
        "Voltage magnitude (pu)": network.buses_t.v_mag_pu.loc[snapshot],
        "Voltage angle (deg)": angles_deg,
    })
    results.index.name = "Bus"
    print(results.to_string(float_format="{:.4f}".format))
    print("Note: linear (DC) power flow assumes every bus is at 1.0 pu and only solves for voltage angles.")

    title("Network summary")
    summary = {
        "Buses": len(network.buses),
        "Lines": len(network.lines),
        "Transformers": len(network.transformers),
        "Generators": len(network.generators),
        "Loads": len(network.loads),
        "Total load (MW)": network.loads.p_set.sum(),
        "Max angle spread (deg)": angles_deg.max() - angles_deg.min(),
    }
    print(f"Buses:      {summary['Buses']}")
    print(f"Lines:      {summary['Lines']}")
    print(f"Generators: {summary['Generators']}")
    return summary


networks = {
    "IEEE 9-bus": load_network("ac_9bus", build_ieee_9bus),
    "IEEE 14-bus": load_network("case14", build_ieee_14bus),
}
summaries = {name: analyze(n) for name, n in networks.items()}

title("Comparison: IEEE 9-bus vs IEEE 14-bus", "#")
comparison = pd.DataFrame(summaries)
comparison["Difference"] = comparison["IEEE 14-bus"] - comparison["IEEE 9-bus"]
comparison.index.name = "Metric"
print(comparison.map(lambda v: f"{v:.0f}" if float(v).is_integer() else f"{v:.1f}").to_string())

s9, s14 = summaries["IEEE 9-bus"], summaries["IEEE 14-bus"]
branches9 = s9["Lines"] + s9["Transformers"]
branches14 = s14["Lines"] + s14["Transformers"]
print(
    f"\nThe 14-bus network is more complex than the 9-bus network:\n"
    f"  - {s14['Buses'] - s9['Buses']} more buses ({s14['Buses']} vs {s9['Buses']})\n"
    f"  - {s14['Lines'] - s9['Lines']} more lines ({s14['Lines']} vs {s9['Lines']}), "
    f"plus {s14['Transformers']} tap-changing transformers the 9-bus case does not have "
    f"({branches14} vs {branches9} branches in total)\n"
    f"  - {s14['Generators'] - s9['Generators']} more generators ({s14['Generators']} vs {s9['Generators']}), "
    f"three of which are synchronous condensers providing voltage support\n"
    f"  - {s14['Loads'] - s9['Loads']} more loads ({s14['Loads']} vs {s9['Loads']}) spread across the network"
)

pypsa 1.2.4 has no examples.ac_9bus(); building it from MATPOWER data.
pypsa 1.2.4 has no examples.case14(); building it from MATPOWER data.

IEEE 9-bus
##########

Buses
=====
       Base voltage (kV)
Bus                     
Bus 1              345.0
Bus 2              345.0
Bus 3              345.0
Bus 4              345.0
Bus 5              345.0
Bus 6              345.0
Bus 7              345.0
Bus 8              345.0
Bus 9              345.0

Transmission lines
         From bus To bus  Resistance (ohm)  Reactance (ohm)
Line                                                       
Line 1-4    Bus 1  Bus 4             0.000           68.558
Line 4-5    Bus 4  Bus 5            20.234          109.503
Line 5-6    Bus 5  Bus 6            46.420          202.343
Line 3-6    Bus 3  Bus 6             0.000           69.749
Line 6-7    Bus 6  Bus 7            14.164          119.977
Line 7-8    Bus 7  Bus 8            10.117           85.698
Line 8-2    Bus 8  Bus 2             0.000      